# ASE calcium imaging tracker: manual one-channel ROI tracking

This notebook tracks manually selected ROIs in a single-channel TIFF movie. It treats each frame as one full image, so there is no green/red split and no red-channel panel.

Main workflow:

1. Set `CHANNEL_NAME`, `ROI_RADIUS`, optional `ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE`, and `save_output` in Cell 2.
2. Load the TIFF. If `ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE` is set, that ImageJ frame is used immediately.
3. If no override is set, preview 20 evenly spaced frames in a single-channel contact sheet, then enter the best ImageJ frame for drawing ROIs.
4. Select ROIs manually from the single-channel view. Toolbar zoom and pan clicks are ignored, so defining a zoom window will not create an ROI.
5. Track ROIs with template matching, motion prediction, peak-ambiguity checks, and QC metrics.
6. Export or display raw ROI intensity values only. dF/F, percent dF/F, candidate detection, confidence plots, and tracked-frame QC overlays are intentionally removed.
7. Display one tracking movie for the selected channel, using outline-only ROI circles.


In [ ]:
# ============================================================
# CELL 1: INSTALL PACKAGES, IMPORT PACKAGES, SET INLINE PLOTTING
# ============================================================

import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tifffile": "tifffile",
    "skimage": "scikit-image",
    "tqdm": "tqdm",
}

missing_packages = []
for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(pip_name)

if len(missing_packages) > 0:
    print("Installing missing packages:")
    print(missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Package installation finished.")
    print("IMPORTANT: restart the kernel after this cell if anything was newly installed.")
else:
    print("All required packages are already installed.")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.widgets import Button
from IPython.display import HTML, display

from tifffile import TiffFile
from skimage.draw import disk
from skimage.feature import match_template
from skimage.filters import gaussian
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

print("Current matplotlib backend:", matplotlib.get_backend())


In [ ]:
# ============================================================
# CELL 2: USER PARAMETERS AND SETTINGS
# ============================================================

# TIFF_PATH = "/Users/sk3526/Desktop/02272026_DCR9229.nd2_green.tif"
TIFF_PATH = "/Users/sk3526/Desktop/02272026_ASE-GCaMP_daf2_e1370.nd2_green.tif"

OUTPUT_DIR = "/Users/sk3526/Desktop"

# Default is False so the notebook does not write files unless you choose to.
save_output = False

# This one-channel notebook analyzes exactly one full-frame channel.
CHANNEL_NAME = "green"
CHANNEL_LABEL = "Green / GCaMP"
CHANNEL_MOVIE_TITLE = "Green"
ACTIVE_CHANNELS = [CHANNEL_NAME]

FPS = 4.0
N_FRAMES_TO_ANALYZE = None

# True reads frames one by one with a waitbar. False keeps the old lazy memmap behavior.
LOAD_TIFF_WITH_PROGRESS = True

# The ROI-selection frame is chosen after loading the TIFF.
# 0.60 means ImageJ frame 600 for a 1000-frame TIFF.
# If you already know the frame you want, set this to an ImageJ frame number.
# When this is not None, Cell 5 skips the contact-sheet preview and Cell 6 skips input().
ROI_SELECTION_FRAME_FRACTION = 0.60
ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE = 181  # Example: 181

# Lightweight preview for choosing the ROI drawing frame when no override is set.
PREVIEW_FRAME_COUNT = 20
PREVIEW_DOWNSAMPLE = 2
PREVIEW_FPS = 2

STIM_FRAMES_IMAGEJ = []
STIM_FRAMES = [f - 1 for f in STIM_FRAMES_IMAGEJ]

START_FRAME_IMAGEJ = 1
END_FRAME_IMAGEJ = None
START_FRAME = START_FRAME_IMAGEJ - 1

# All ROIs in this one channel use this radius.
ROI_RADIUS = 2
ROI_DIAMETER = 2 * ROI_RADIUS

# Tracking settings.
# Search/template radii are computed from each ROI radius, so larger ROIs get
# larger local context while small ROIs cannot search too far away.
SEARCH_RADIUS_MULTIPLIER = 3.0
MIN_SEARCH_RADIUS = 3
MAX_SEARCH_RADIUS = None
TEMPLATE_RADIUS_MULTIPLIER = 5.0
MIN_TEMPLATE_RADIUS = 8
MAX_TEMPLATE_RADIUS = None
GAUSSIAN_SIGMA_FOR_TRACKING = 1.0
TRACKING_BACKGROUND_SUBTRACT = True
TRACKING_BACKGROUND_SIGMA = 8.0
TRACKING_USE_CONTEXT_MASK = True
TRACKING_ROI_WEIGHT = 1.0
TRACKING_CONTEXT_WEIGHT = 0.35
TRACKING_ROBUST_NORMALIZE = True
TRACKING_EPSILON = 1e-6
MOTION_PREDICTION_WEIGHT = 0.25
DISTANCE_PENALTY_WEIGHT = 0.12
SUBPIXEL_PEAK_REFINEMENT = True
MAX_DISPLACEMENT_ALLOWED = 6
MIN_TEMPLATE_MATCH_SCORE = 0.35
MIN_PEAK_MARGIN = 0.05
PEAK_EXCLUSION_RADIUS = 4
HOLD_POSITION_ON_BAD_TRACK = True
# Leave template updates off for the safest run. If tracking QC is stable,
# set this True to adapt templates only on high-confidence matches.
UPDATE_TEMPLATE_IF_GOOD = False
TEMPLATE_UPDATE_ALPHA = 0.20
MIN_SCORE_FOR_TEMPLATE_UPDATE = 0.70
MIN_MARGIN_FOR_TEMPLATE_UPDATE = 0.10
# If a ROI gets held for multiple frames, briefly expand the search window.
EXPAND_SEARCH_ON_BAD_TRACK = True
BAD_TRACK_STREAK_BEFORE_EXPAND = 2
EXPANDED_SEARCH_RADIUS_MULTIPLIER = 2.0
MAX_EXPANDED_SEARCH_RADIUS = 25
# Trajectory and neighbor-geometry QC catches jumps that match score alone misses.
TRAJECTORY_JUMP_WARNING_MULTIPLIER = 1.5
TRAJECTORY_DRIFT_WARNING_MULTIPLIER = 4.0
NEIGHBOR_DISTANCE_CHANGE_WARNING_FRACTION = 0.35
MIN_ROIS_FOR_NEIGHBOR_QC = 3
MIN_ACCEPTED_FRACTION_WARNING = 0.85
MAX_BAD_TRACK_STREAK_WARNING = 5

# Brightest-pixel snapping is off because it can jump to a nearby wrong neuron.
RECENTER_ON_BRIGHTEST_PIXEL = False
RECENTER_SEARCH_RADIUS = None  # Not used unless RECENTER_ON_BRIGHTEST_PIXEL is True.
RECENTER_GAUSSIAN_SIGMA = 0.75

# Manual ROI selection display.
# "tk" opens only the clickable ROI selector in an external window.
# The notebook switches back to inline plotting after ROI selection.
MANUAL_CLICK_BACKEND = "tk"
ALLOW_TEXT_ROI_FALLBACK = True

# Static grid settings for coordinate reading.
MAJOR_TICK_STEP = 50
MINOR_TICK_STEP = 10

CHANNEL_LABELS = {CHANNEL_NAME: CHANNEL_LABEL}
CHANNEL_MOVIE_TITLES = {CHANNEL_NAME: CHANNEL_MOVIE_TITLE}

# Display-only brightness/contrast settings for preview, ROI selection, and movies.
# These do not change the raw intensity values used for analysis.
# DISPLAY_HIGH_PERCENTILE controls the white point; higher values usually darken and saturate less.
# DISPLAY_GAMMA < 1 brightens midtones; DISPLAY_GAMMA > 1 darkens midtones.
# DISPLAY_OUTPUT_FLOOR lifts true-black pixels; keep it at 0.0 if you want black pixels to stay black.
DISPLAY_LOW_PERCENTILE = {CHANNEL_NAME: 1.0}
DISPLAY_HIGH_PERCENTILE = {CHANNEL_NAME: 99.7}
DISPLAY_GAMMA = {CHANNEL_NAME: 1.3}
DISPLAY_OUTPUT_FLOOR = {CHANNEL_NAME: 0.0}
CHANNEL_COLORS = {CHANNEL_NAME: "lime"}
CHANNEL_ROI_DIAMETERS = {CHANNEL_NAME: float(ROI_DIAMETER)}
CHANNEL_ROI_RADII = {channel: diameter / 2 for channel, diameter in CHANNEL_ROI_DIAMETERS.items()}

if ROI_DIAMETER <= 0:
    raise ValueError("ROI diameter must be positive.")

if save_output:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print(
    f"Setup: one channel={CHANNEL_NAME}, "
    f"ROI radius={ROI_RADIUS}px, "
    f"ROI frame override={ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE}, "
    f"save_output={save_output}"
)


In [ ]:
# ============================================================
# CELL 3: LOAD TIFF LAZILY
# ============================================================

tif = TiffFile(TIFF_PATH)
series = tif.series[0]


def load_series_with_progress(series):
    pages = list(series.pages)
    if len(pages) <= 1:
        raise ValueError("series does not expose one TIFF page per movie frame")

    first_frame = np.squeeze(pages[0].asarray())
    if first_frame.ndim != 2:
        raise ValueError(f"Expected each TIFF page to be 2D after squeeze, got {first_frame.shape}")

    loaded = np.empty((len(pages), *first_frame.shape), dtype=first_frame.dtype)
    loaded[0] = first_frame

    with tqdm(total=len(pages), desc="Loading TIFF frames", unit="frame") as pbar:
        pbar.update(1)
        for frame_idx, page in enumerate(pages[1:], start=1):
            loaded[frame_idx] = np.squeeze(page.asarray())
            pbar.update(1)

    return loaded


if LOAD_TIFF_WITH_PROGRESS:
    try:
        movie = load_series_with_progress(series)
    except Exception as err:
        print("Progress loading was not available for this TIFF; falling back to lazy memmap.")
        print("Reason:", err)
        movie = series.asarray(out="memmap")
else:
    print("Opening TIFF as a lazy memmap; frames are read when accessed, so no frame-by-frame waitbar is shown.")
    movie = series.asarray(out="memmap")

movie = np.squeeze(movie)

if movie.ndim != 3:
    raise ValueError(f"Expected movie shape [frames, height, width], but got {movie.shape}")

n_total_frames, frame_h, frame_w = movie.shape

if N_FRAMES_TO_ANALYZE is None:
    n_frames = n_total_frames
else:
    n_frames = min(N_FRAMES_TO_ANALYZE, n_total_frames)

if END_FRAME_IMAGEJ is None:
    END_FRAME = n_frames - 1
else:
    END_FRAME = min(END_FRAME_IMAGEJ - 1, n_frames - 1)

if START_FRAME < 0 or START_FRAME > END_FRAME:
    raise ValueError("START_FRAME_IMAGEJ and END_FRAME_IMAGEJ define an invalid analysis range.")

DEFAULT_ROI_SELECTION_FRAME_IMAGEJ = int(round(n_frames * ROI_SELECTION_FRAME_FRACTION))
DEFAULT_ROI_SELECTION_FRAME_IMAGEJ = int(np.clip(DEFAULT_ROI_SELECTION_FRAME_IMAGEJ, 1, n_frames))

if ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE is None:
    ROI_SELECTION_FRAME_IMAGEJ = DEFAULT_ROI_SELECTION_FRAME_IMAGEJ
    ROI_SELECTION_FRAME_SOURCE = "default fraction"
else:
    ROI_SELECTION_FRAME_IMAGEJ = int(np.clip(int(ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE), 1, n_frames))
    ROI_SELECTION_FRAME_SOURCE = "Cell 2 override"

ROI_SELECTION_FRAME = ROI_SELECTION_FRAME_IMAGEJ - 1

print(
    f"Loaded {n_total_frames} one-channel frames ({frame_h} x {frame_w}); "
    f"analyzing ImageJ frames {START_FRAME + 1}-{END_FRAME + 1}; "
    f"ROI frame: ImageJ {ROI_SELECTION_FRAME_IMAGEJ} ({ROI_SELECTION_FRAME_SOURCE})."
)


In [ ]:
# ============================================================
# CELL 4: BASIC IMAGE, DISPLAY, AND ROI FUNCTIONS
# ============================================================


def get_frame(frame_idx):
    return movie[frame_idx].astype(np.float32)


def get_channel_frame(frame_idx, channel=None):
    if channel is None:
        channel = CHANNEL_NAME
    channel = str(channel).lower()
    if channel != CHANNEL_NAME:
        raise ValueError(f"This notebook only has the {CHANNEL_NAME!r} channel.")
    return get_frame(frame_idx)


def get_green_frame(frame_idx):
    return get_channel_frame(frame_idx, CHANNEL_NAME)


def display_setting(setting, channel, default):
    if isinstance(setting, dict):
        return setting.get(channel, default)
    return setting


def normalize_for_display(img, channel=None, p_low=None, p_high=None, gamma=None, output_floor=None):
    if channel is None:
        channel = CHANNEL_NAME
    if p_low is None:
        p_low = display_setting(DISPLAY_LOW_PERCENTILE, channel, 0.0)
    if p_high is None:
        p_high = display_setting(DISPLAY_HIGH_PERCENTILE, channel, 99.5)
    if gamma is None:
        gamma = display_setting(DISPLAY_GAMMA, channel, 1.0)
    if output_floor is None:
        output_floor = display_setting(DISPLAY_OUTPUT_FLOOR, channel, 0.0)

    lo, hi = np.nanpercentile(img, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(img, dtype=np.float32)

    out = (img.astype(np.float32) - lo) / (hi - lo)
    out = np.clip(out, 0, 1)

    if gamma is not None and gamma > 0 and gamma != 1:
        out = np.power(out, gamma)

    if output_floor is not None and output_floor > 0:
        output_floor = float(np.clip(output_floor, 0, 0.95))
        out = output_floor + (1 - output_floor) * out

    return out


def show_display_tuning_grid(frame_idx=None, channels=None):
    if frame_idx is None:
        frame_idx = ROI_SELECTION_FRAME
    if channels is None:
        channels = ACTIVE_CHANNELS

    presets = [
        ("current", None, None, None, None),
        ("darker", 1.0, 99.7, 1.3, 0.0),
        ("linear", 1.0, 99.7, 1.0, 0.0),
        ("brighter dim", 0.0, 99.5, 0.7, 0.0),
    ]

    fig, axes = plt.subplots(
        len(channels),
        len(presets),
        figsize=(4.0 * len(presets), 4.0 * len(channels)),
        squeeze=False,
    )

    for row_idx, channel in enumerate(channels):
        img = get_channel_frame(frame_idx, channel)
        for col_idx, (name, p_low, p_high, gamma, output_floor) in enumerate(presets):
            ax = axes[row_idx, col_idx]
            if name == "current":
                display_img = normalize_for_display(img, channel=channel)
                p_low = display_setting(DISPLAY_LOW_PERCENTILE, channel, 0.0)
                p_high = display_setting(DISPLAY_HIGH_PERCENTILE, channel, 99.5)
                gamma = display_setting(DISPLAY_GAMMA, channel, 1.0)
                output_floor = display_setting(DISPLAY_OUTPUT_FLOOR, channel, 0.0)
            else:
                display_img = normalize_for_display(
                    img,
                    channel=channel,
                    p_low=p_low,
                    p_high=p_high,
                    gamma=gamma,
                    output_floor=output_floor,
                )
            ax.imshow(display_img, cmap="gray", vmin=0, vmax=1, aspect="equal")
            ax.set_title(
                f"{name}\nlow={p_low}, high={p_high}\ngamma={gamma}, floor={output_floor}",
                fontsize=9,
            )
            ax.set_axis_off()

    fig.suptitle(f"Display tuning preview: ImageJ frame {frame_idx + 1}", fontsize=12)
    plt.tight_layout()
    plt.show()


def add_coordinate_grid(ax, img_shape, channel=None):
    h, w = img_shape
    ax.set_xlim(0, w - 1)
    ax.set_ylim(h - 1, 0)
    ax.set_xticks(np.arange(0, w, MAJOR_TICK_STEP))
    ax.set_yticks(np.arange(0, h, MAJOR_TICK_STEP))
    ax.set_xticks(np.arange(0, w, MINOR_TICK_STEP), minor=True)
    ax.set_yticks(np.arange(0, h, MINOR_TICK_STEP), minor=True)
    ax.grid(which="major", linewidth=0.7, alpha=0.45)
    ax.grid(which="minor", linewidth=0.3, alpha=0.20)
    ax.set_xlabel("x coordinate")
    ax.set_ylabel("y coordinate")


def roi_diameter_for_channel(channel):
    return CHANNEL_ROI_DIAMETERS[channel]


def roi_radius_for_channel(channel):
    return CHANNEL_ROI_RADII[channel]


def roi_label(channel, roi_id):
    return f"{channel[0].upper()}{int(roi_id)}"


def validate_channel_coordinate(channel, x, y):
    img = get_channel_frame(ROI_SELECTION_FRAME, channel)
    h, w = img.shape
    if not (0 <= x < w):
        print(f"Warning: x={x:.2f} is outside image width 0 to {w - 1}.")
    if not (0 <= y < h):
        print(f"Warning: y={y:.2f} is outside image height 0 to {h - 1}.")


def parse_xy_string(s):
    parts = s.replace(" ", "").split(",")
    if len(parts) != 2:
        raise ValueError("Please enter coordinates as x,y")
    return float(parts[0]), float(parts[1])


def parse_many_xy_string(s):
    points = []
    for chunk in s.split(";"):
        chunk = chunk.strip()
        if len(chunk) > 0:
            points.append(parse_xy_string(chunk))
    return points


def circular_mask_values(img, x, y, diameter):
    radius = float(diameter) / 2
    rr, cc = disk((float(y), float(x)), radius, shape=img.shape)
    return img[rr, cc]


def extract_raw_roi_intensity(channel_img, x, y, diameter):
    roi_vals = circular_mask_values(channel_img, x, y, diameter)
    raw_mean = float(np.nanmean(roi_vals)) if len(roi_vals) > 0 else np.nan
    raw_median = float(np.nanmedian(roi_vals)) if len(roi_vals) > 0 else np.nan
    return raw_mean, raw_median, int(len(roi_vals))


def show_channel_frame(frame_idx, title_extra=""):
    img = get_channel_frame(frame_idx, CHANNEL_NAME)
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 6.5))
    ax.imshow(normalize_for_display(img, channel=CHANNEL_NAME), cmap="gray")
    title = f"{CHANNEL_LABELS[CHANNEL_NAME]}\nImageJ frame {frame_idx + 1}"
    if title_extra:
        title += "\n" + title_extra
    ax.set_title(title)
    add_coordinate_grid(ax, img.shape, channel=CHANNEL_NAME)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CELL 5: PREVIEW 20 EVENLY SPACED FRAMES
# ============================================================
#
# This cell only displays a lightweight contact sheet when no ROI frame override
# was set in Cell 2. Choose your preferred ImageJ frame number from the titles,
# then enter it in the next cell.
# ============================================================


def get_preview_frame_image(frame_idx):
    img = get_channel_frame(frame_idx, CHANNEL_NAME)
    if PREVIEW_DOWNSAMPLE > 1:
        img = img[::PREVIEW_DOWNSAMPLE, ::PREVIEW_DOWNSAMPLE]
    return normalize_for_display(img, channel=CHANNEL_NAME)


def show_roi_frame_preview_contact_sheet():
    global PREVIEW_IMAGEJ_FRAMES

    PREVIEW_IMAGEJ_FRAMES = np.unique(
        np.round(np.linspace(1, n_frames, PREVIEW_FRAME_COUNT)).astype(int)
    )
    PREVIEW_IMAGEJ_FRAMES = PREVIEW_IMAGEJ_FRAMES[
        (PREVIEW_IMAGEJ_FRAMES >= 1) & (PREVIEW_IMAGEJ_FRAMES <= n_frames)
    ]
    preview_python_frames = PREVIEW_IMAGEJ_FRAMES - 1

    n_cols = 5
    n_rows = int(np.ceil(len(preview_python_frames) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows), squeeze=False)

    for ax in axes.ravel():
        ax.set_axis_off()

    for ax, imagej_frame, frame_idx in zip(axes.ravel(), PREVIEW_IMAGEJ_FRAMES, preview_python_frames):
        ax.imshow(get_preview_frame_image(int(frame_idx)), cmap="gray", vmin=0, vmax=1, aspect="equal")
        ax.set_title(f"ImageJ {int(imagej_frame)}", fontsize=10)

    fig.suptitle(f"Preview frames for ROI selection: {CHANNEL_MOVIE_TITLE}", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    print("Preview ImageJ frames:", [int(x) for x in PREVIEW_IMAGEJ_FRAMES])
    print("Default ROI selection frame:", int(DEFAULT_ROI_SELECTION_FRAME_IMAGEJ))


if ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE is None:
    show_roi_frame_preview_contact_sheet()
else:
    PREVIEW_IMAGEJ_FRAMES = np.array([], dtype=int)
    print(
        "ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE is set in Cell 2; "
        f"skipping preview and using ImageJ frame {ROI_SELECTION_FRAME_IMAGEJ}."
    )


In [ ]:
# ============================================================
# CELL 6: CHOOSE ROI SELECTION FRAME
# ============================================================
#
# If ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE was set in Cell 2, this cell uses it
# directly and does not prompt. Otherwise, enter a previewed ImageJ frame number
# or press Enter to keep the default frame.
# ============================================================


def choose_roi_selection_frame():
    global ROI_SELECTION_FRAME_IMAGEJ, ROI_SELECTION_FRAME, ROI_SELECTION_FRAME_SOURCE

    default_frame = int(DEFAULT_ROI_SELECTION_FRAME_IMAGEJ)

    if ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE is None:
        user_text = input(
            f"Enter ImageJ frame for ROI selection, or press Enter for default {default_frame}: "
        ).strip()
        if len(user_text) > 0:
            chosen_frame = int(user_text)
            ROI_SELECTION_FRAME_SOURCE = "manual input"
        else:
            chosen_frame = default_frame
            ROI_SELECTION_FRAME_SOURCE = "default fraction"
    else:
        chosen_frame = int(ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE)
        ROI_SELECTION_FRAME_SOURCE = "Cell 2 override"

    ROI_SELECTION_FRAME_IMAGEJ = int(np.clip(chosen_frame, 1, n_frames))
    ROI_SELECTION_FRAME = ROI_SELECTION_FRAME_IMAGEJ - 1

    print(f"ROI selection frame set to ImageJ {ROI_SELECTION_FRAME_IMAGEJ} ({ROI_SELECTION_FRAME_SOURCE}).")


choose_roi_selection_frame()


In [ ]:
# ============================================================
# CELL 7: MANUAL ONE-CHANNEL ROI SELECTION
# ============================================================
#
# Use the external interactive matplotlib window to click ROI centers.
# Toolbar zoom/pan clicks are ignored and will not create ROIs.
# Use the Home button to return to the full view, then zoom elsewhere.
# Press Done or ENTER/RETURN when finished. Later plots are restored to inline notebook output.
# ============================================================

ROI_TABLE_COLUMNS = [
    "channel",
    "roi_id",
    "x_initial",
    "y_initial",
    "frame_initial_python",
    "frame_initial_imagej",
    "roi_diameter",
    "roi_radius",
    "source",
]


def make_roi_tables_from_points(points_by_channel, frame_idx, source="manual_click"):
    roi_tables = {}
    for channel in ACTIVE_CHANNELS:
        rows = []
        for i, (x, y) in enumerate(points_by_channel.get(channel, []), start=1):
            validate_channel_coordinate(channel, x, y)
            rows.append({
                "channel": channel,
                "roi_id": int(i),
                "x_initial": float(x),
                "y_initial": float(y),
                "frame_initial_python": int(frame_idx),
                "frame_initial_imagej": int(frame_idx + 1),
                "roi_diameter": float(roi_diameter_for_channel(channel)),
                "roi_radius": float(roi_radius_for_channel(channel)),
                "source": source,
            })
        roi_tables[channel] = pd.DataFrame(rows, columns=ROI_TABLE_COLUMNS)
    return roi_tables


def toolbar_is_active(fig):
    toolbar = getattr(fig.canvas, "toolbar", None)
    mode = getattr(toolbar, "mode", "") if toolbar is not None else ""
    if mode is None:
        return False
    mode_text = str(mode)
    return mode_text not in {"", "None", "_Mode.NONE"}


def select_rois_one_channel(frame_idx, backend="tk"):
    channel = CHANNEL_NAME
    selected_points = {channel: []}
    artist_history = []

    try:
        get_ipython().run_line_magic("matplotlib", backend)
    except Exception as err:
        print("Could not switch matplotlib backend.")
        print("Error:", err)

    done = {"value": False}
    buttons = []

    fig, ax = plt.subplots(1, 1, figsize=(7.5, 8))
    plt.subplots_adjust(bottom=0.16)

    img = get_channel_frame(frame_idx, channel)
    display_img = normalize_for_display(img, channel=channel)
    ax.imshow(display_img, cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.set_title(
        f"{CHANNEL_MOVIE_TITLES[channel]}\nImageJ frame {frame_idx + 1}",
        fontsize=11,
    )
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, img.shape[1] - 1)
    ax.set_ylim(img.shape[0] - 1, 0)
    ax.set_xlabel("x coordinate")
    ax.set_ylabel("y coordinate")
    full_limits = ((0, img.shape[1] - 1), (img.shape[0] - 1, 0))

    def redraw():
        fig.canvas.draw_idle()

    def draw_roi(x, y):
        roi_id = len(selected_points[channel])
        radius = roi_radius_for_channel(channel)
        color = CHANNEL_COLORS[channel]

        circle = plt.Circle(
            (x, y),
            radius,
            fill=False,
            linewidth=1.7,
            edgecolor=color,
        )
        ax.add_patch(circle)

        label = ax.text(
            x + radius + 2,
            y,
            roi_label(channel, roi_id),
            color=color,
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1),
            clip_on=True,
        )

        artist_history.append({"artists": [circle, label]})
        redraw()

    def add_roi(x, y):
        selected_points[channel].append((float(x), float(y)))
        draw_roi(float(x), float(y))

    def undo_last(event=None):
        if len(artist_history) == 0:
            return
        entry = artist_history.pop()
        if len(selected_points[channel]) > 0:
            selected_points[channel].pop()
        for artist in entry["artists"]:
            artist.remove()
        redraw()

    def clear_all(event=None):
        while len(artist_history) > 0:
            entry = artist_history.pop()
            for artist in entry["artists"]:
                artist.remove()
        selected_points[channel] = []
        redraw()

    def home_view(event=None):
        xlim, ylim = full_limits
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        redraw()

    def finish(event=None):
        done["value"] = True

    def on_click(event):
        if event.button != 1:
            return
        if event.inaxes is not ax:
            return
        if event.xdata is None or event.ydata is None:
            return
        if toolbar_is_active(fig):
            return

        add_roi(event.xdata, event.ydata)

    def on_key(event):
        if event.key in {"enter", "return"}:
            finish(event)
        elif event.key in {"backspace", "delete", "u"}:
            undo_last(event)
        elif event.key in {"h", "home"}:
            home_view(event)

    def on_close(event):
        done["value"] = True

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    fig.canvas.mpl_connect("close_event", on_close)

    button_specs = [
        ("Home", [0.20, 0.04, 0.13, 0.05], home_view),
        ("Undo", [0.36, 0.04, 0.13, 0.05], undo_last),
        ("Clear", [0.52, 0.04, 0.13, 0.05], clear_all),
        ("Done", [0.68, 0.04, 0.13, 0.05], finish),
    ]
    for label, rect, callback in button_specs:
        button_ax = fig.add_axes(rect)
        button = Button(button_ax, label)
        button.on_clicked(callback)
        buttons.append(button)

    try:
        plt.show(block=False)
        while (not done["value"]) and plt.fignum_exists(fig.number):
            fig.canvas.flush_events()
            time.sleep(0.05)
    except Exception as err:
        print("Manual clicking did not work in this environment.")
        print("Error:", err)
    finally:
        try:
            fig.canvas.flush_events()
        except Exception:
            pass

    return make_roi_tables_from_points(selected_points, frame_idx, source="manual_click")


def enter_rois_by_text(frame_idx):
    points_by_channel = {channel: [] for channel in ACTIVE_CHANNELS}
    print("Text fallback: enter ROI centers as x,y; x,y. Leave blank for no ROIs.")
    for channel in ACTIVE_CHANNELS:
        text = input(f"Enter {channel} ROI centers: ").strip()
        if len(text) > 0:
            points_by_channel[channel] = parse_many_xy_string(text)
    return make_roi_tables_from_points(points_by_channel, frame_idx, source="manual_text")


roi_tables = select_rois_one_channel(
    ROI_SELECTION_FRAME,
    backend=MANUAL_CLICK_BACKEND,
)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

total_selected = sum(len(df) for df in roi_tables.values())
if total_selected == 0 and ALLOW_TEXT_ROI_FALLBACK:
    roi_tables = enter_rois_by_text(ROI_SELECTION_FRAME)
    total_selected = sum(len(df) for df in roi_tables.values())

if total_selected == 0:
    raise RuntimeError("No ROIs were selected.")

roi_counts = {channel: len(roi_tables[channel]) for channel in ACTIVE_CHANNELS}
print("Selected ROIs:", roi_counts)

if save_output:
    for channel in ACTIVE_CHANNELS:
        out_path = os.path.join(OUTPUT_DIR, f"selected_{channel}_rois.csv")
        roi_tables[channel].to_csv(out_path, index=False)


In [ ]:
# ============================================================
# CELL 8: VISUALIZE SELECTED ROIS
# ============================================================


def plot_selected_rois(frame_idx, roi_tables):
    channel = CHANNEL_NAME
    img = get_channel_frame(frame_idx, channel)
    fig, ax = plt.subplots(1, 1, figsize=(7.5, 8))

    ax.imshow(normalize_for_display(img, channel=channel), cmap="gray", vmin=0, vmax=1, aspect="equal")
    ax.set_title(f"{CHANNEL_MOVIE_TITLES[channel]}\nImageJ frame {frame_idx + 1}")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, img.shape[1] - 1)
    ax.set_ylim(img.shape[0] - 1, 0)
    ax.set_xlabel("x coordinate")
    ax.set_ylabel("y coordinate")

    df = roi_tables.get(channel, pd.DataFrame())
    for _, row in df.iterrows():
        roi_id = int(row["roi_id"])
        x = float(row["x_initial"])
        y = float(row["y_initial"])
        radius = float(row["roi_radius"])
        color = CHANNEL_COLORS[channel]

        circle = plt.Circle(
            (x, y),
            radius,
            fill=False,
            linewidth=1.7,
            edgecolor=color,
        )
        ax.add_patch(circle)
        ax.text(
            x + radius + 2,
            y,
            roi_label(channel, roi_id),
            color=color,
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1),
            clip_on=True,
        )

    plt.tight_layout()
    plt.show()


plot_selected_rois(ROI_SELECTION_FRAME, roi_tables)


In [ ]:
# ============================================================
# CELL 9: TRACKING HELPER FUNCTIONS
# ============================================================


def crop_with_bounds(img, x, y, radius):
    h, w = img.shape
    x = int(round(x))
    y = int(round(y))
    radius = int(np.ceil(radius))
    x0 = max(0, x - radius)
    x1 = min(w, x + radius + 1)
    y0 = max(0, y - radius)
    y1 = min(h, y + radius + 1)
    crop = img[y0:y1, x0:x1]
    return crop, x0, x1, y0, y1


def preprocess_for_tracking(channel_img):
    work = channel_img.astype(np.float32, copy=False)

    if GAUSSIAN_SIGMA_FOR_TRACKING is not None and GAUSSIAN_SIGMA_FOR_TRACKING > 0:
        work = gaussian(work, sigma=GAUSSIAN_SIGMA_FOR_TRACKING, preserve_range=True)

    if TRACKING_BACKGROUND_SUBTRACT:
        background = gaussian(work, sigma=TRACKING_BACKGROUND_SIGMA, preserve_range=True)
        work = work - background

    if TRACKING_ROBUST_NORMALIZE:
        median = np.nanmedian(work)
        mad = np.nanmedian(np.abs(work - median))
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale <= TRACKING_EPSILON:
            scale = np.nanstd(work)
        if np.isfinite(scale) and scale > TRACKING_EPSILON:
            work = (work - median) / scale
        else:
            work = work - median

    return np.nan_to_num(work, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def bounded_tracking_radius(value, min_value=None, max_value=None):
    radius = float(value)
    if min_value is not None:
        radius = max(radius, float(min_value))
    if max_value is not None:
        radius = min(radius, float(max_value))
    return int(np.ceil(radius))


def tracking_search_radius_for_roi(roi_diameter):
    roi_radius = float(roi_diameter) / 2
    return bounded_tracking_radius(
        roi_radius * SEARCH_RADIUS_MULTIPLIER,
        min_value=MIN_SEARCH_RADIUS,
        max_value=MAX_SEARCH_RADIUS,
    )


def tracking_template_radius_for_roi(roi_diameter):
    roi_radius = float(roi_diameter) / 2
    return bounded_tracking_radius(
        roi_radius * TEMPLATE_RADIUS_MULTIPLIER,
        min_value=MIN_TEMPLATE_RADIUS,
        max_value=MAX_TEMPLATE_RADIUS,
    )


def make_template_weight_mask(template_shape, roi_radius):
    if not TRACKING_USE_CONTEXT_MASK:
        return np.ones(template_shape, dtype=np.float32)

    h, w = template_shape
    yy, xx = np.indices(template_shape)
    cx = (w - 1) / 2
    cy = (h - 1) / 2
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    outer_radius = min(cx, cy)

    weights = np.zeros(template_shape, dtype=np.float32)
    weights[dist <= outer_radius] = float(TRACKING_CONTEXT_WEIGHT)
    weights[dist <= float(roi_radius)] = float(TRACKING_ROI_WEIGHT)
    return weights


def make_template(channel_img, x, y, template_radius, roi_radius=None):
    tracking_img = preprocess_for_tracking(channel_img)
    template, x0, x1, y0, y1 = crop_with_bounds(tracking_img, x, y, template_radius)
    if roi_radius is not None:
        template = template * make_template_weight_mask(template.shape, roi_radius)
    return template


def estimate_subpixel_peak(result_for_peak, y_peak, x_peak):
    if not SUBPIXEL_PEAK_REFINEMENT:
        return float(y_peak), float(x_peak)

    y_sub = float(y_peak)
    x_sub = float(x_peak)

    if 0 < x_peak < result_for_peak.shape[1] - 1:
        left = result_for_peak[y_peak, x_peak - 1]
        center = result_for_peak[y_peak, x_peak]
        right = result_for_peak[y_peak, x_peak + 1]
        denom = left - 2 * center + right
        if np.isfinite(denom) and abs(denom) > TRACKING_EPSILON:
            x_sub += float(np.clip(0.5 * (left - right) / denom, -0.5, 0.5))

    if 0 < y_peak < result_for_peak.shape[0] - 1:
        top = result_for_peak[y_peak - 1, x_peak]
        center = result_for_peak[y_peak, x_peak]
        bottom = result_for_peak[y_peak + 1, x_peak]
        denom = top - 2 * center + bottom
        if np.isfinite(denom) and abs(denom) > TRACKING_EPSILON:
            y_sub += float(np.clip(0.5 * (top - bottom) / denom, -0.5, 0.5))

    return y_sub, x_sub


def compute_second_best_score(result_for_peak, y_peak, x_peak):
    if PEAK_EXCLUSION_RADIUS is None or PEAK_EXCLUSION_RADIUS <= 0:
        exclusion_radius = 1
    else:
        exclusion_radius = int(np.ceil(PEAK_EXCLUSION_RADIUS))

    masked = np.array(result_for_peak, copy=True)
    yy, xx = np.indices(masked.shape)
    peak_mask = (xx - x_peak) ** 2 + (yy - y_peak) ** 2 <= exclusion_radius ** 2
    masked[peak_mask] = -np.inf

    if not np.any(np.isfinite(masked)):
        return np.nan

    return float(np.nanmax(masked))


def empty_tracking_metrics(reason):
    return {
        "accepted": False,
        "match_score": np.nan,
        "second_best_score": np.nan,
        "peak_margin": np.nan,
        "displacement": np.nan,
        "rejection_reason": reason,
    }


def track_one_step(channel_img, prev_x, prev_y, template, template_radius, search_radius, search_center_x=None, search_center_y=None):
    if template.size == 0:
        return float(prev_x), float(prev_y), empty_tracking_metrics("empty_template")

    if search_center_x is None:
        search_center_x = prev_x
    if search_center_y is None:
        search_center_y = prev_y

    tracking_img = preprocess_for_tracking(channel_img)
    search_crop_radius = int(np.ceil(template_radius + search_radius))
    search_crop, sx0, sx1, sy0, sy1 = crop_with_bounds(
        tracking_img,
        search_center_x,
        search_center_y,
        search_crop_radius,
    )

    if search_crop.shape[0] < template.shape[0] or search_crop.shape[1] < template.shape[1]:
        return float(prev_x), float(prev_y), empty_tracking_metrics("search_crop_smaller_than_template")

    try:
        result = match_template(search_crop, template, pad_input=False)
    except Exception as err:
        return float(prev_x), float(prev_y), empty_tracking_metrics(f"match_template_failed: {err}")

    if not np.any(np.isfinite(result)):
        return float(prev_x), float(prev_y), empty_tracking_metrics("no_finite_match_scores")

    result_for_peak = np.where(np.isfinite(result), result, -np.inf)
    template_h, template_w = template.shape

    if DISTANCE_PENALTY_WEIGHT is not None and DISTANCE_PENALTY_WEIGHT > 0:
        yy, xx = np.indices(result_for_peak.shape)
        candidate_x = sx0 + xx + (template_w - 1) / 2
        candidate_y = sy0 + yy + (template_h - 1) / 2
        distance_to_prediction = np.sqrt((candidate_x - search_center_x) ** 2 + (candidate_y - search_center_y) ** 2)
        search_scale = max(float(search_radius), 1.0)
        penalty = float(DISTANCE_PENALTY_WEIGHT) * (distance_to_prediction / search_scale) ** 2
        score_for_peak = result_for_peak - penalty
        score_for_peak = np.where(np.isfinite(result_for_peak), score_for_peak, -np.inf)
    else:
        score_for_peak = result_for_peak

    y_peak, x_peak = np.unravel_index(np.argmax(score_for_peak), score_for_peak.shape)
    match_score = float(result_for_peak[y_peak, x_peak])
    weighted_match_score = float(score_for_peak[y_peak, x_peak])
    second_best_score = compute_second_best_score(score_for_peak, y_peak, x_peak)
    peak_margin = weighted_match_score - second_best_score if np.isfinite(second_best_score) else np.nan

    y_sub, x_sub = estimate_subpixel_peak(result_for_peak, int(y_peak), int(x_peak))
    new_x = sx0 + x_sub + (template_w - 1) / 2
    new_y = sy0 + y_sub + (template_h - 1) / 2

    displacement = float(np.sqrt((new_x - prev_x) ** 2 + (new_y - prev_y) ** 2))

    rejection_reasons = []
    if displacement > MAX_DISPLACEMENT_ALLOWED:
        rejection_reasons.append("large_displacement")
    if MIN_TEMPLATE_MATCH_SCORE is not None and match_score < MIN_TEMPLATE_MATCH_SCORE:
        rejection_reasons.append("low_match_score")
    if (
        MIN_PEAK_MARGIN is not None and
        np.isfinite(peak_margin) and
        peak_margin < MIN_PEAK_MARGIN
    ):
        rejection_reasons.append("ambiguous_peak")

    accepted = len(rejection_reasons) == 0
    metrics = {
        "accepted": bool(accepted),
        "match_score": match_score,
        "second_best_score": second_best_score,
        "peak_margin": peak_margin,
        "displacement": displacement,
        "rejection_reason": ";".join(rejection_reasons),
    }

    return float(new_x), float(new_y), metrics


def blend_templates(old_template, new_template):
    if old_template.shape != new_template.shape:
        return new_template
    alpha = float(np.clip(TEMPLATE_UPDATE_ALPHA, 0, 1))
    return (1 - alpha) * old_template + alpha * new_template


def recenter_on_brightest_pixel(channel_img, x, y, search_radius):
    if search_radius is None or search_radius <= 0:
        return float(x), float(y)

    crop, x0, x1, y0, y1 = crop_with_bounds(channel_img, x, y, int(np.ceil(search_radius)))
    if crop.size == 0:
        return float(x), float(y)

    if RECENTER_GAUSSIAN_SIGMA and RECENTER_GAUSSIAN_SIGMA > 0:
        search_img = gaussian(crop, sigma=RECENTER_GAUSSIAN_SIGMA, preserve_range=True)
    else:
        search_img = crop

    yy, xx = np.indices(search_img.shape)
    cx = float(x) - x0
    cy = float(y) - y0
    circular_search_mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= float(search_radius) ** 2
    search_img = np.where(circular_search_mask & np.isfinite(search_img), search_img, -np.inf)

    if not np.any(np.isfinite(search_img)):
        return float(x), float(y)

    y_peak, x_peak = np.unravel_index(np.argmax(search_img), search_img.shape)
    return float(x0 + x_peak), float(y0 + y_peak)


In [ ]:
# ============================================================
# CELL 10: BUILD INITIAL ROI POSITIONS FOR TRACKING
# ============================================================

INITIAL_POSITION_COLUMNS = [
    "channel",
    "roi_id",
    "frame",
    "frame_imagej",
    "x",
    "y",
    "roi_diameter",
    "roi_radius",
    "source",
]


def build_initial_positions_df(roi_tables):
    rows = []

    for channel, roi_df in roi_tables.items():
        if len(roi_df) == 0:
            continue
        for _, row in roi_df.iterrows():
            rows.append({
                "channel": channel,
                "roi_id": int(row["roi_id"]),
                "frame": int(row["frame_initial_python"]),
                "frame_imagej": int(row["frame_initial_imagej"]),
                "x": float(row["x_initial"]),
                "y": float(row["y_initial"]),
                "roi_diameter": float(row["roi_diameter"]),
                "roi_radius": float(row["roi_radius"]),
                "source": "initial",
            })

    initial_positions_df = pd.DataFrame(rows, columns=INITIAL_POSITION_COLUMNS)
    if len(initial_positions_df) == 0:
        raise RuntimeError("No ROI positions were found.")

    initial_positions_df = initial_positions_df.sort_values(["channel", "roi_id"]).reset_index(drop=True)

    for _, row in initial_positions_df.iterrows():
        validate_channel_coordinate(row["channel"], row["x"], row["y"])

    if save_output:
        initial_positions_path = os.path.join(OUTPUT_DIR, "initial_roi_positions.csv")
        initial_positions_df.to_csv(initial_positions_path, index=False)

    return initial_positions_df


initial_positions_df = build_initial_positions_df(roi_tables)


In [ ]:
# ============================================================
# CELL 11: ROI TRACKING FUNCTIONS
# ============================================================

TRACE_COLUMNS = [
    "channel",
    "frame",
    "frame_imagej",
    "time_sec",
    "roi_id",
    "x",
    "y",
    "roi_diameter",
    "roi_radius",
    "tracking_template_radius",
    "tracking_search_radius",
    "effective_search_radius",
    "raw_mean",
    "raw_median",
    "n_pixels",
    "tracking_direction",
    "segment_start_frame",
    "segment_start_imagej",
    "tracking_status",
    "accepted",
    "match_score",
    "second_best_score",
    "peak_margin",
    "displacement",
    "bad_track_streak",
    "rejection_reason",
]


def track_roi_segment(channel, roi_id, start_frame, end_frame, start_x, start_y, roi_diameter, direction):
    if direction not in [+1, -1]:
        raise ValueError("direction must be +1 or -1")

    if direction == +1 and end_frame < start_frame:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    if direction == -1 and end_frame > start_frame:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    start_img = get_channel_frame(start_frame, channel)
    if RECENTER_ON_BRIGHTEST_PIXEL:
        search_radius = RECENTER_SEARCH_RADIUS
        if search_radius is None:
            search_radius = roi_diameter / 2
        start_x, start_y = recenter_on_brightest_pixel(
            start_img,
            start_x,
            start_y,
            search_radius,
        )

    roi_radius = float(roi_diameter) / 2
    template_radius = tracking_template_radius_for_roi(roi_diameter)
    tracking_search_radius = tracking_search_radius_for_roi(roi_diameter)
    template = make_template(start_img, start_x, start_y, template_radius, roi_radius=roi_radius)

    current_x = float(start_x)
    current_y = float(start_y)
    last_dx = 0.0
    last_dy = 0.0
    bad_track_streak = 0
    rows = []
    frame_range = range(start_frame, end_frame + direction, direction)
    motion_weight = 0.0 if MOTION_PREDICTION_WEIGHT is None else float(MOTION_PREDICTION_WEIGHT)

    for frame_idx in frame_range:
        channel_img = get_channel_frame(frame_idx, channel)
        tracking_status = "anchor"
        metrics = {
            "accepted": True,
            "match_score": np.nan,
            "second_best_score": np.nan,
            "peak_margin": np.nan,
            "displacement": 0.0,
            "rejection_reason": "",
        }
        effective_search_radius = tracking_search_radius
        if EXPAND_SEARCH_ON_BAD_TRACK and bad_track_streak >= BAD_TRACK_STREAK_BEFORE_EXPAND:
            effective_search_radius = int(np.ceil(tracking_search_radius * EXPANDED_SEARCH_RADIUS_MULTIPLIER))
            if MAX_EXPANDED_SEARCH_RADIUS is not None:
                effective_search_radius = min(effective_search_radius, int(MAX_EXPANDED_SEARCH_RADIUS))

        if frame_idx != start_frame:
            prev_x = current_x
            prev_y = current_y
            predicted_x = current_x + motion_weight * last_dx
            predicted_y = current_y + motion_weight * last_dy
            x_new, y_new, metrics = track_one_step(
                channel_img,
                current_x,
                current_y,
                template,
                template_radius=template_radius,
                search_radius=effective_search_radius,
                search_center_x=predicted_x,
                search_center_y=predicted_y,
            )

            if metrics["accepted"]:
                current_x = float(x_new)
                current_y = float(y_new)
                last_dx = current_x - prev_x
                last_dy = current_y - prev_y
                bad_track_streak = 0
                tracking_status = "accepted"

                should_update_template = UPDATE_TEMPLATE_IF_GOOD
                if MIN_SCORE_FOR_TEMPLATE_UPDATE is not None:
                    should_update_template = should_update_template and metrics["match_score"] >= MIN_SCORE_FOR_TEMPLATE_UPDATE
                if MIN_MARGIN_FOR_TEMPLATE_UPDATE is not None:
                    should_update_template = should_update_template and np.isfinite(metrics["peak_margin"]) and metrics["peak_margin"] >= MIN_MARGIN_FOR_TEMPLATE_UPDATE

                if should_update_template:
                    new_template = make_template(channel_img, current_x, current_y, template_radius, roi_radius=roi_radius)
                    template = blend_templates(template, new_template)
            else:
                bad_track_streak += 1
                last_dx = 0.0
                last_dy = 0.0
                if HOLD_POSITION_ON_BAD_TRACK:
                    current_x = prev_x
                    current_y = prev_y
                    tracking_status = "held"
                else:
                    current_x = float(x_new)
                    current_y = float(y_new)
                    tracking_status = "rejected_moved"

        if RECENTER_ON_BRIGHTEST_PIXEL:
            search_radius = RECENTER_SEARCH_RADIUS
            if search_radius is None:
                search_radius = roi_diameter / 2
            current_x, current_y = recenter_on_brightest_pixel(
                channel_img,
                current_x,
                current_y,
                search_radius,
            )

        raw_mean, raw_median, n_pixels = extract_raw_roi_intensity(
            channel_img,
            current_x,
            current_y,
            roi_diameter,
        )

        rows.append({
            "channel": channel,
            "frame": int(frame_idx),
            "frame_imagej": int(frame_idx + 1),
            "time_sec": float(frame_idx / FPS),
            "roi_id": int(roi_id),
            "x": float(current_x),
            "y": float(current_y),
            "roi_diameter": float(roi_diameter),
            "roi_radius": float(roi_diameter / 2),
            "tracking_template_radius": int(template_radius),
            "tracking_search_radius": int(tracking_search_radius),
            "effective_search_radius": int(effective_search_radius),
            "raw_mean": raw_mean,
            "raw_median": raw_median,
            "n_pixels": int(n_pixels),
            "tracking_direction": "forward" if direction == +1 else "backward",
            "segment_start_frame": int(start_frame),
            "segment_start_imagej": int(start_frame + 1),
            "tracking_status": tracking_status,
            "accepted": bool(metrics["accepted"]),
            "match_score": metrics["match_score"],
            "second_best_score": metrics["second_best_score"],
            "peak_margin": metrics["peak_margin"],
            "displacement": metrics["displacement"],
            "bad_track_streak": int(bad_track_streak),
            "rejection_reason": metrics["rejection_reason"],
        })

    return pd.DataFrame(rows, columns=TRACE_COLUMNS)


def track_single_roi(initial_row, start_frame, end_frame):
    channel = str(initial_row["channel"])
    roi_id = int(initial_row["roi_id"])
    initial_frame = int(initial_row["frame"])
    initial_x = float(initial_row["x"])
    initial_y = float(initial_row["y"])
    roi_diameter = float(initial_row["roi_diameter"])
    segments = []

    if initial_frame >= start_frame:
        segments.append(track_roi_segment(
            channel=channel,
            roi_id=roi_id,
            start_frame=initial_frame,
            end_frame=start_frame,
            start_x=initial_x,
            start_y=initial_y,
            roi_diameter=roi_diameter,
            direction=-1,
        ))

    if end_frame >= initial_frame:
        segments.append(track_roi_segment(
            channel=channel,
            roi_id=roi_id,
            start_frame=initial_frame,
            end_frame=end_frame,
            start_x=initial_x,
            start_y=initial_y,
            roi_diameter=roi_diameter,
            direction=+1,
        ))

    roi_trace = pd.concat(segments, ignore_index=True)
    roi_trace = roi_trace[
        (roi_trace["frame"] >= start_frame) &
        (roi_trace["frame"] <= end_frame)
    ]
    roi_trace = (
        roi_trace
        .sort_values(["channel", "roi_id", "frame", "tracking_direction"])
        .drop_duplicates(subset=["channel", "roi_id", "frame"], keep="last")
        .sort_values("frame")
        .reset_index(drop=True)
    )

    return roi_trace


def track_all_rois(initial_positions_df, start_frame, end_frame):
    all_traces = []

    for _, initial_row in tqdm(initial_positions_df.iterrows(), total=len(initial_positions_df), desc="Tracking ROIs"):
        all_traces.append(track_single_roi(initial_row, start_frame=start_frame, end_frame=end_frame))

    if len(all_traces) == 0:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    traces_df = pd.concat(all_traces, ignore_index=True)
    traces_df = traces_df.sort_values(["channel", "roi_id", "frame"]).reset_index(drop=True)
    return traces_df


In [ ]:
# ============================================================
# CELL 12: RUN TRACKING AND PREPARE RAW ROI INTENSITY TABLES
# ============================================================

traces_df = track_all_rois(
    initial_positions_df=initial_positions_df,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
)

channel_traces = {
    channel: traces_df[traces_df["channel"] == channel].copy().reset_index(drop=True)
    for channel in ACTIVE_CHANNELS
}


def make_wide_raw_table(channel_df, channel, value_col="raw_mean"):
    if len(channel_df) == 0:
        return pd.DataFrame()

    times = (
        channel_df[["frame_imagej", "time_sec"]]
        .drop_duplicates()
        .sort_values("frame_imagej")
    )
    wide = channel_df.pivot(index="frame_imagej", columns="roi_id", values=value_col)
    wide.columns = [roi_label(channel, c) for c in wide.columns]
    wide = wide.reset_index()
    wide = times.merge(wide, on="frame_imagej", how="left")
    return wide


def finite_median(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    return float(np.median(values))


def finite_max(values, default=np.nan):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return default
    return float(np.max(values))


def append_qc_note(existing_notes, new_note):
    existing_notes = str(existing_notes) if pd.notna(existing_notes) else ""
    notes = [note.strip() for note in existing_notes.split(" | ") if len(note.strip()) > 0]
    if new_note not in notes:
        notes.append(new_note)
    return " | ".join(notes)


def max_actual_step_distance(sub):
    sub = sub.sort_values("frame")
    if len(sub) < 2:
        return 0.0
    dx = sub["x"].astype(float).diff().values
    dy = sub["y"].astype(float).diff().values
    step_dist = np.sqrt(dx ** 2 + dy ** 2)
    return finite_max(step_dist, default=0.0)


def max_distance_from_anchor(sub):
    if len(sub) == 0:
        return 0.0
    anchor_frame = int(sub["segment_start_frame"].iloc[0])
    anchor_rows = sub[sub["frame"] == anchor_frame]
    if len(anchor_rows) == 0:
        anchor_rows = sub.iloc[[0]]
    anchor_x = float(anchor_rows["x"].iloc[0])
    anchor_y = float(anchor_rows["y"].iloc[0])
    dist = np.sqrt((sub["x"].astype(float).values - anchor_x) ** 2 + (sub["y"].astype(float).values - anchor_y) ** 2)
    return finite_max(dist, default=0.0)


def add_neighbor_distance_qc(summary_df, traces):
    if len(summary_df) == 0 or len(traces) == 0:
        return summary_df

    summary_df = summary_df.copy()
    summary_df["max_neighbor_distance_change_fraction"] = np.nan

    for channel, channel_traces in traces.groupby("channel"):
        roi_ids = sorted(channel_traces["roi_id"].unique())
        if len(roi_ids) < MIN_ROIS_FOR_NEIGHBOR_QC:
            continue

        anchor_frame = int(channel_traces["segment_start_frame"].mode().iloc[0])
        anchor_df = channel_traces[channel_traces["frame"] == anchor_frame].drop_duplicates("roi_id")
        if len(anchor_df) < MIN_ROIS_FOR_NEIGHBOR_QC:
            continue

        anchor_positions = {
            int(row["roi_id"]): np.array([float(row["x"]), float(row["y"])])
            for _, row in anchor_df.iterrows()
        }
        baseline_neighbor_distance = {}
        for roi_id, pos in anchor_positions.items():
            other_distances = [
                np.linalg.norm(pos - other_pos)
                for other_id, other_pos in anchor_positions.items()
                if other_id != roi_id
            ]
            if len(other_distances) > 0:
                baseline_neighbor_distance[roi_id] = float(np.min(other_distances))

        max_change_by_roi = {roi_id: 0.0 for roi_id in baseline_neighbor_distance}
        for _, frame_df in channel_traces.groupby("frame"):
            frame_df = frame_df.drop_duplicates("roi_id")
            if len(frame_df) < 2:
                continue
            frame_positions = {
                int(row["roi_id"]): np.array([float(row["x"]), float(row["y"])])
                for _, row in frame_df.iterrows()
            }
            for roi_id, baseline_distance in baseline_neighbor_distance.items():
                if roi_id not in frame_positions or baseline_distance <= TRACKING_EPSILON:
                    continue
                pos = frame_positions[roi_id]
                current_distances = [
                    np.linalg.norm(pos - other_pos)
                    for other_id, other_pos in frame_positions.items()
                    if other_id != roi_id
                ]
                if len(current_distances) == 0:
                    continue
                current_distance = float(np.min(current_distances))
                change_fraction = abs(current_distance - baseline_distance) / baseline_distance
                max_change_by_roi[roi_id] = max(max_change_by_roi[roi_id], change_fraction)

        for roi_id, max_change in max_change_by_roi.items():
            row_mask = (summary_df["channel"] == channel) & (summary_df["roi_id"] == roi_id)
            if not row_mask.any():
                continue
            summary_df.loc[row_mask, "max_neighbor_distance_change_fraction"] = float(max_change)
            if max_change > NEIGHBOR_DISTANCE_CHANGE_WARNING_FRACTION:
                summary_df.loc[row_mask, "qc_flag"] = True
                summary_df.loc[row_mask, "qc_notes"] = summary_df.loc[row_mask, "qc_notes"].apply(
                    lambda notes: append_qc_note(notes, "neighbor_geometry_change")
                )

    return summary_df


def summarize_tracking_quality(traces):
    records = []
    if len(traces) == 0:
        return pd.DataFrame()

    for (channel, roi_id), sub in traces.groupby(["channel", "roi_id"]):
        sub = sub.sort_values("frame")
        step_rows = sub[sub["tracking_status"] != "anchor"]
        n_steps = int(len(step_rows))
        n_accepted = int(step_rows["accepted"].sum()) if n_steps > 0 else 0
        accepted_fraction = float(n_accepted / n_steps) if n_steps > 0 else 1.0
        max_bad_track_streak = int(sub["bad_track_streak"].max()) if len(sub) > 0 else 0
        base_search_radius = finite_median(sub["tracking_search_radius"]) if "tracking_search_radius" in sub.columns else np.nan
        if not np.isfinite(base_search_radius) or base_search_radius <= 0:
            base_search_radius = 1.0
        max_actual_step_px = max_actual_step_distance(sub)
        max_distance_from_anchor_px = max_distance_from_anchor(sub)

        notes = []
        if accepted_fraction < MIN_ACCEPTED_FRACTION_WARNING:
            notes.append("low_acceptance")
        if max_bad_track_streak >= MAX_BAD_TRACK_STREAK_WARNING:
            notes.append("long_bad_streak")
        if max_actual_step_px > TRAJECTORY_JUMP_WARNING_MULTIPLIER * base_search_radius:
            notes.append("trajectory_jump")
        if max_distance_from_anchor_px > TRAJECTORY_DRIFT_WARNING_MULTIPLIER * base_search_radius:
            notes.append("large_anchor_drift")

        records.append({
            "channel": channel,
            "roi_id": int(roi_id),
            "roi_label": roi_label(channel, roi_id),
            "n_tracked_frames": int(len(sub)),
            "n_tracking_steps": n_steps,
            "accepted_steps": n_accepted,
            "accepted_fraction": accepted_fraction,
            "median_match_score": finite_median(step_rows["match_score"]),
            "median_peak_margin": finite_median(step_rows["peak_margin"]),
            "median_displacement_px": finite_median(step_rows["displacement"]),
            "max_displacement_px": float(np.nanmax(step_rows["displacement"])) if n_steps > 0 else 0.0,
            "max_actual_step_px": max_actual_step_px,
            "max_distance_from_anchor_px": max_distance_from_anchor_px,
            "base_search_radius_px": base_search_radius,
            "max_bad_track_streak": max_bad_track_streak,
            "qc_flag": len(notes) > 0,
            "qc_notes": " | ".join(notes),
        })

    summary_df = pd.DataFrame(records)
    summary_df = add_neighbor_distance_qc(summary_df, traces)
    return summary_df


def plot_tracking_qc_summary(summary_df):
    if len(summary_df) == 0:
        return

    plot_df = summary_df.sort_values(["channel", "roi_id"]).reset_index(drop=True)
    labels = plot_df["roi_label"].tolist()
    y = np.arange(len(labels))
    fig_height = max(3.0, 0.42 * len(labels) + 1.8)
    fig, axes = plt.subplots(1, 2, figsize=(11, fig_height), sharey=True)

    axes[0].barh(y, plot_df["accepted_fraction"], color="seagreen", alpha=0.85)
    axes[0].axvline(MIN_ACCEPTED_FRACTION_WARNING, color="black", linestyle="--", linewidth=1)
    axes[0].set_xlim(0, 1.02)
    axes[0].set_xlabel("Accepted tracking fraction")
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(labels)
    axes[0].invert_yaxis()

    axes[1].barh(y, plot_df["max_bad_track_streak"], color="darkorange", alpha=0.85)
    axes[1].axvline(MAX_BAD_TRACK_STREAK_WARNING, color="black", linestyle="--", linewidth=1)
    axes[1].set_xlabel("Longest rejected-step streak")

    fig.suptitle("Tracking QC by ROI", fontsize=12)
    plt.tight_layout()
    plt.show()


raw_mean_tables = {}
raw_median_tables = {}

for channel in ACTIVE_CHANNELS:
    channel_df = channel_traces[channel]
    raw_mean_tables[channel] = make_wide_raw_table(channel_df, channel, value_col="raw_mean")
    raw_median_tables[channel] = make_wide_raw_table(channel_df, channel, value_col="raw_median")

    if save_output and len(channel_df) > 0:
        long_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_roi_traces_long.csv")
        mean_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_mean_by_roi.csv")
        median_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_median_by_roi.csv")

        channel_df.to_csv(long_path, index=False)
        raw_mean_tables[channel].to_csv(mean_path, index=False)
        raw_median_tables[channel].to_csv(median_path, index=False)

tracking_qc_summary = summarize_tracking_quality(traces_df)
display(tracking_qc_summary)
plot_tracking_qc_summary(tracking_qc_summary)

flagged_qc = tracking_qc_summary[tracking_qc_summary["qc_flag"]] if len(tracking_qc_summary) > 0 else pd.DataFrame()
if len(flagged_qc) > 0:
    print("Tracking QC warning: inspect these ROIs before trusting their traces:")
    display_cols = [
        "channel",
        "roi_id",
        "roi_label",
        "accepted_fraction",
        "max_bad_track_streak",
        "max_actual_step_px",
        "max_distance_from_anchor_px",
        "max_neighbor_distance_change_fraction",
        "qc_notes",
    ]
    display(flagged_qc[[col for col in display_cols if col in flagged_qc.columns]])
else:
    print("Tracking QC passed for all ROIs using the current warning thresholds.")

if save_output and len(tracking_qc_summary) > 0:
    qc_path = os.path.join(OUTPUT_DIR, "tracking_qc_summary.csv")
    tracking_qc_summary.to_csv(qc_path, index=False)

print("Tracking complete. Raw tables are in raw_mean_tables and raw_median_tables.")


In [ ]:
# ============================================================
# CELL 13: OPTIONAL RAW AND NORMALIZED INTENSITY TRACE PLOTS
# ============================================================
#
# This plots raw intensity on the left and normalized intensity on the right.
# The x-axis is restricted to the analyzed/tracked frame range, so stimulus
# markers outside the analyzed movie do not create empty space.
# Set PLOT_TRACE_PANELS = False if you only want the tables from Cell 12.
# ============================================================

PLOT_TRACE_PANELS = True
TRACE_VALUE_COL = "raw_mean"

# Normalized panel settings.
# "dff" plots (F - F0) / F0. F0 is the per-ROI percentile below.
# "fold" plots F / F0. "max" plots F / max(F). "zscore" plots standard score per ROI.
NORMALIZATION_MODE = "max"
NORMALIZATION_F0_PERCENTILE = 20
NORMALIZATION_BASELINE_IMAGEJ = None  # Example: (1, 80); None uses the full trace.
NORMALIZATION_EPSILON = 1e-6


def trace_time_limits(channel_df):
    if len(channel_df) == 0:
        return 0, 0
    frame_min = int(channel_df["frame"].min())
    frame_max = int(channel_df["frame"].max())
    return frame_min / FPS, frame_max / FPS


def baseline_values_for_normalization(sub, value_col):
    baseline = sub.copy()
    if NORMALIZATION_BASELINE_IMAGEJ is not None:
        start_imagej, end_imagej = NORMALIZATION_BASELINE_IMAGEJ
        baseline = baseline[
            (baseline["frame_imagej"] >= start_imagej) &
            (baseline["frame_imagej"] <= end_imagej)
        ]
        if len(baseline) == 0:
            baseline = sub.copy()

    values = baseline[value_col].astype(float).values
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan

    f0 = float(np.nanpercentile(values, NORMALIZATION_F0_PERCENTILE))
    sigma = float(np.nanstd(values))
    return f0, sigma


def normalize_trace_values(sub, value_col):
    values = sub[value_col].astype(float).values
    f0, sigma = baseline_values_for_normalization(sub, value_col)

    if NORMALIZATION_MODE == "dff":
        if not np.isfinite(f0) or abs(f0) <= NORMALIZATION_EPSILON:
            return np.full_like(values, np.nan, dtype=float), "dF/F0"
        return (values - f0) / f0, "dF/F0"

    if NORMALIZATION_MODE == "fold":
        if not np.isfinite(f0) or abs(f0) <= NORMALIZATION_EPSILON:
            return np.full_like(values, np.nan, dtype=float), "F/F0"
        return values / f0, "F/F0"

    if NORMALIZATION_MODE == "max":
        finite_values = values[np.isfinite(values)]
        if len(finite_values) == 0:
            return np.full_like(values, np.nan, dtype=float), "F/max(F)"
        max_value = float(np.nanmax(finite_values))
        if not np.isfinite(max_value) or abs(max_value) <= NORMALIZATION_EPSILON:
            return np.full_like(values, np.nan, dtype=float), "F/max(F)"
        return values / max_value, "F/max(F)"

    if NORMALIZATION_MODE == "zscore":
        if not np.isfinite(sigma) or sigma <= NORMALIZATION_EPSILON:
            return np.full_like(values, np.nan, dtype=float), "z-score"
        return (values - f0) / sigma, "z-score"

    raise ValueError('NORMALIZATION_MODE must be "dff", "fold", "max", or "zscore".')


def add_in_range_stim_lines(ax, stim_frames, x_min, x_max):
    if stim_frames is None:
        return
    for sf in stim_frames:
        stim_time = sf / FPS
        if x_min <= stim_time <= x_max:
            ax.axvline(stim_time, linestyle="--", linewidth=1)


def plot_raw_and_normalized_traces(channel_df, channel, value_col="raw_mean", stim_frames=None):
    if len(channel_df) == 0:
        return

    roi_ids = sorted(channel_df["roi_id"].unique())
    x_min, x_max = trace_time_limits(channel_df)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
    ax_raw, ax_norm = axes
    norm_ylabel = None

    for roi_id in roi_ids:
        sub = channel_df[channel_df["roi_id"] == roi_id]
        label = roi_label(channel, roi_id)
        ax_raw.plot(sub["time_sec"], sub[value_col], label=label, linewidth=1)
        norm_values, norm_ylabel = normalize_trace_values(sub, value_col)
        ax_norm.plot(sub["time_sec"], norm_values, label=label, linewidth=1)

    for ax in axes:
        add_in_range_stim_lines(ax, stim_frames, x_min, x_max)
        ax.set_xlim(x_min, x_max)
        ax.set_xlabel("Time (s)")

    ax_raw.set_ylabel(value_col)
    ax_raw.set_title(f"{CHANNEL_MOVIE_TITLES[channel]} raw ROI intensity")
    ax_norm.set_ylabel(norm_ylabel if norm_ylabel is not None else NORMALIZATION_MODE)
    ax_norm.set_title(f"{CHANNEL_MOVIE_TITLES[channel]} normalized ROI intensity")
    ax_norm.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


if PLOT_TRACE_PANELS:
    for channel in ACTIVE_CHANNELS:
        plot_raw_and_normalized_traces(channel_traces[channel], channel, value_col=TRACE_VALUE_COL, stim_frames=STIM_FRAMES)


In [ ]:
# ============================================================
# CELL 14: IMAGESC-STYLE RAW INTENSITY HEATMAPS
# ============================================================
#
# These are additional heatmap views of the same raw mean intensity traces.
# Each row is one ROI, and each column is one time point/frame.
# ============================================================

PLOT_RAW_IMAGESC = True
IMAGESC_VALUE_COL = "raw_mean"
IMAGESC_LOW_PERCENTILE = 1
IMAGESC_HIGH_PERCENTILE = 99
IMAGESC_CMAP = "viridis"


def make_trace_matrix(channel_df, channel, value_col="raw_mean"):
    if len(channel_df) == 0:
        return np.empty((0, 0)), [], np.array([])

    frames = np.array(sorted(channel_df["frame"].unique()))
    times = (
        channel_df[["frame", "time_sec"]]
        .drop_duplicates()
        .set_index("frame")
        .loc[frames, "time_sec"]
        .values
    )
    roi_ids = sorted(channel_df["roi_id"].unique())
    matrix = np.full((len(roi_ids), len(frames)), np.nan, dtype=float)

    frame_to_col = {frame: idx for idx, frame in enumerate(frames)}
    roi_to_row = {roi_id: idx for idx, roi_id in enumerate(roi_ids)}

    for _, row in channel_df.iterrows():
        matrix[roi_to_row[row["roi_id"]], frame_to_col[row["frame"]]] = row[value_col]

    labels = [roi_label(channel, roi_id) for roi_id in roi_ids]
    return matrix, labels, times


def plot_raw_imagesc(channel_df, channel, value_col="raw_mean", stim_frames=None):
    matrix, labels, times = make_trace_matrix(channel_df, channel, value_col=value_col)
    if matrix.size == 0:
        return

    finite_values = matrix[np.isfinite(matrix)]
    if len(finite_values) == 0:
        return

    vmin, vmax = np.nanpercentile(finite_values, [IMAGESC_LOW_PERCENTILE, IMAGESC_HIGH_PERCENTILE])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(finite_values), np.nanmax(finite_values)

    fig_height = max(2.8, 0.45 * len(labels) + 1.8)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    extent = [times[0], times[-1], len(labels) + 0.5, 0.5]
    im = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        cmap=IMAGESC_CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=extent,
    )

    if stim_frames is not None:
        for sf in stim_frames:
            ax.axvline(sf / FPS, linestyle="--", linewidth=1, color="white", alpha=0.9)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("ROI")
    ax.set_yticks(np.arange(1, len(labels) + 1))
    ax.set_yticklabels(labels)
    ax.set_title(f"{CHANNEL_MOVIE_TITLES[channel]} raw ROI intensity imagesc")

    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(value_col)

    plt.tight_layout()
    plt.show()


if PLOT_RAW_IMAGESC:
    for channel in ACTIVE_CHANNELS:
        plot_raw_imagesc(channel_traces[channel], channel, value_col=IMAGESC_VALUE_COL, stim_frames=STIM_FRAMES)


In [ ]:
# ============================================================
# CELL 15: INTERACTIVE TRACKING MOVIE WITH OUTLINE-ONLY ROIS
# ============================================================
#
# This shows the tracked one-channel movie with outline-only ROI circles.
# Optional MP4 saving only happens when both SAVE_MP4 and save_output are True.
# ============================================================

mpl.rcParams["animation.embed_limit"] = 300  # MB

# ---------------- USER SETTINGS ----------------
MOVIE_FRAME_STEP = 10
MOVIE_FPS = 5
MOVIE_DPI = 150

MOVIE_START_FRAME_IMAGEJ = 1
MOVIE_END_FRAME_IMAGEJ = n_frames

CROP_TO_TRACK_REGION = True
CROP_MARGIN = 30

SHOW_TRAIL = False
TRAIL_LENGTH_FRAMES = 200

SAVE_MP4 = False
MP4_FILENAME = f"roi_tracking_movie_every_{MOVIE_FRAME_STEP}_frames.mp4"
# ------------------------------------------------

movie_channels = [
    channel for channel in ACTIVE_CHANNELS
    if channel in channel_traces and len(channel_traces[channel]) > 0
]

if len(movie_channels) == 0:
    raise RuntimeError("No tracked channels are available for the movie.")

movie_start = max(0, MOVIE_START_FRAME_IMAGEJ - 1)
movie_end = min(n_frames - 1, MOVIE_END_FRAME_IMAGEJ - 1)
frame_list = list(range(movie_start, movie_end + 1, MOVIE_FRAME_STEP))

if len(frame_list) == 0:
    raise ValueError("No frames selected for movie. Check frame range settings.")

channel_height, channel_width = get_channel_frame(movie_start, movie_channels[0]).shape

if CROP_TO_TRACK_REGION:
    crop_source = pd.concat([channel_traces[channel] for channel in movie_channels], ignore_index=True)
    x_all = crop_source["x"].values
    y_all = crop_source["y"].values

    x_min = max(0, int(np.floor(np.nanmin(x_all) - CROP_MARGIN)))
    x_max = min(channel_width - 1, int(np.ceil(np.nanmax(x_all) + CROP_MARGIN)))

    y_min = max(0, int(np.floor(np.nanmin(y_all) - CROP_MARGIN)))
    y_max = min(channel_height - 1, int(np.ceil(np.nanmax(y_all) + CROP_MARGIN)))
else:
    x_min, x_max = 0, channel_width - 1
    y_min, y_max = 0, channel_height - 1


def get_display_channel_crop(frame_idx, channel):
    img = get_channel_frame(frame_idx, channel)
    crop = img[y_min:y_max + 1, x_min:x_max + 1]
    return normalize_for_display(crop, channel=channel)


first_frame = frame_list[0]
n_cols = len(movie_channels)
fig, axes = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 6), squeeze=False)
axes = axes[0]

image_artists = {}
title_artists = {}
circle_artists = {}
text_artists = {}
trail_artists = {}
roi_ids_by_channel = {}

for ax, channel in zip(axes, movie_channels):
    first_img = get_display_channel_crop(first_frame, channel)
    image_artists[channel] = ax.imshow(first_img, cmap="gray", vmin=0, vmax=1, aspect="equal")
    title_artists[channel] = ax.set_title(CHANNEL_MOVIE_TITLES[channel], fontsize=12)
    ax.set_xlim(0, first_img.shape[1] - 1)
    ax.set_ylim(first_img.shape[0] - 1, 0)
    ax.set_xlabel("x coordinate, cropped view")
    ax.set_ylabel("y coordinate")

    roi_ids = sorted(channel_traces[channel]["roi_id"].unique())
    roi_ids_by_channel[channel] = roi_ids
    circle_artists[channel] = {}
    text_artists[channel] = {}
    trail_artists[channel] = {}

    cmap = plt.get_cmap("tab10")
    for k, roi_id in enumerate(roi_ids):
        color = cmap(k % 10)
        radius = roi_radius_for_channel(channel)

        circ = plt.Circle(
            (0, 0),
            radius,
            fill=False,
            lw=1.8,
            color=color,
            visible=False,
        )
        ax.add_patch(circ)

        label_text = ax.text(
            0,
            0,
            roi_label(channel, roi_id),
            color=color,
            fontsize=8,
            visible=False,
            bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1),
            clip_on=True,
        )

        trail_line, = ax.plot(
            [],
            [],
            "-",
            lw=1.0,
            color=color,
            alpha=0.75,
            visible=False,
        )

        circle_artists[channel][roi_id] = circ
        text_artists[channel][roi_id] = label_text
        trail_artists[channel][roi_id] = trail_line

fig_title = fig.suptitle("", fontsize=12, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])


def update_movie(frame_i):
    frame_idx = frame_list[frame_i]
    fig_title.set_text(f"ImageJ frame {frame_idx + 1} / {n_frames}")

    artists = [fig_title]

    for channel in movie_channels:
        image_artists[channel].set_data(get_display_channel_crop(frame_idx, channel))
        title_artists[channel].set_text(CHANNEL_MOVIE_TITLES[channel])
        artists.extend([image_artists[channel], title_artists[channel]])

        sub_now = channel_traces[channel][channel_traces[channel]["frame"] == frame_idx].copy()

        for roi_id in roi_ids_by_channel[channel]:
            circ = circle_artists[channel][roi_id]
            label_text = text_artists[channel][roi_id]
            trail_line = trail_artists[channel][roi_id]

            sub_roi = sub_now[sub_now["roi_id"] == roi_id]
            if len(sub_roi) == 0:
                circ.set_visible(False)
                label_text.set_visible(False)
                trail_line.set_visible(False)
                continue

            x = float(sub_roi["x"].iloc[0])
            y = float(sub_roi["y"].iloc[0])
            radius = roi_radius_for_channel(channel)

            xc = x - x_min
            yc = y - y_min

            circ.center = (xc, yc)
            circ.set_visible(True)

            label_text.set_position((xc + radius + 2, yc))
            label_text.set_text(roi_label(channel, roi_id))
            label_text.set_visible(True)

            if SHOW_TRAIL:
                sub_hist = channel_traces[channel][
                    (channel_traces[channel]["roi_id"] == roi_id) &
                    (channel_traces[channel]["frame"] <= frame_idx) &
                    (channel_traces[channel]["frame"] >= frame_idx - TRAIL_LENGTH_FRAMES)
                ].copy().sort_values("frame")

                if len(sub_hist) >= 2:
                    xh = sub_hist["x"].values - x_min
                    yh = sub_hist["y"].values - y_min
                    trail_line.set_data(xh, yh)
                    trail_line.set_visible(True)
                else:
                    trail_line.set_visible(False)
            else:
                trail_line.set_visible(False)

            artists.extend([circ, label_text, trail_line])

    return artists


anim_roi_tracking = FuncAnimation(
    fig,
    update_movie,
    frames=len(frame_list),
    interval=1000 / MOVIE_FPS,
    blit=False,
    cache_frame_data=False,
)

if SAVE_MP4:
    if save_output:
        movie_output_dir = Path(OUTPUT_DIR)
        movie_output_dir.mkdir(parents=True, exist_ok=True)
        mp4_path = movie_output_dir / MP4_FILENAME
        writer = FFMpegWriter(fps=MOVIE_FPS)
        anim_roi_tracking.save(mp4_path, writer=writer, dpi=MOVIE_DPI)
        print("Saved mp4:", mp4_path)
    else:
        print("SAVE_MP4 is True, but save_output is False. Set save_output = True in Cell 2 to write the movie file.")

plt.close(fig)
display(HTML(anim_roi_tracking.to_jshtml()))
